# 23. 🧰 Прикладное: Фурье, свёртки и обработка сигналов

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justxor/math-for-ai-2026/blob/main/notebooks/23_fourier_signals.ipynb)

Код из раздела [modules/23_fourier_signals.md](https://github.com/justxor/math-for-ai-2026/blob/main/modules/23_fourier_signals.md#m23). Запускайте ячейки сверху вниз: последующие используют переменные предыдущих. Теория, формулы и картинки — в тексте курса по ссылке.

In [ ]:
# Colab: всё нужное уже установлено. Локально: pip install -r requirements.txt
import numpy as np
import math
np.set_printoptions(precision=4, suppress=True)

### 🏋️ Практика модуля

**23.1.** Найдите доминирующие частоты в сигнале с помощью FFT.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
import numpy as np
fs = 1000; t = np.arange(0, 1, 1 / fs)
x = np.sin(2 * np.pi * 50 * t) + 0.5 * np.sin(2 * np.pi * 120 * t)
amp = np.abs(np.fft.rfft(x)) * 2 / t.size; f = np.fft.rfftfreq(t.size, 1 / fs)
print(f[np.argsort(-amp)[:2]], amp[np.argsort(-amp)[:2]].round(2))   # [50. 120.] [1.  0.5]

### 🏋️ Практика модуля

**23.2.** Проверьте теорему о свёртке численно.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
a, b = np.random.randn(64), np.random.randn(16)
direct = np.convolve(a, b)                                   # длина 64 + 16 − 1 = 79
n = len(a) + len(b) - 1
via_fft = np.fft.irfft(np.fft.rfft(a, n) * np.fft.rfft(b, n), n)
print(np.allclose(direct, via_fft))                          # True

## 🔬 Лаборатория: иллюстрации модуля

Это код картинок из курса. Меняйте параметры (learning rate, температуру, число точек, ядро…) и перезапускайте ячейку — так интуиция появляется быстрее всего.

In [ ]:
# Помощники для иллюстраций (те же, что в scripts/make_figures.py)
import matplotlib.pyplot as plt
from scipy import stats
C = ["#2563eb", "#dc2626", "#059669", "#d97706", "#7c3aed", "#0891b2"]
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def save(fig, name):          # в ноутбуке просто показываем картинку
    fig.tight_layout(); plt.show()

def arrow(ax, v, color, label=None, origin=(0, 0), **kw):
    ax.annotate("", xy=(origin[0] + v[0], origin[1] + v[1]), xytext=origin,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2, **kw))
    if label:
        ax.text(origin[0] + v[0] * 1.08, origin[1] + v[1] * 1.08, label, color=color, fontsize=13, fontweight="bold")

def rosen_like(x, y):
    return 0.5 * x**2 + 4 * y**2

In [ ]:
def fourier():
    fs = 1000; t = np.arange(0, 1, 1 / fs)
    rng = np.random.default_rng(0)
    sig = 1.0 * np.sin(2 * np.pi * 50 * t) + 0.5 * np.sin(2 * np.pi * 120 * t) + 0.3 * rng.normal(size=t.size)
    F = np.fft.rfft(sig); freqs = np.fft.rfftfreq(t.size, 1 / fs)
    fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
    axes[0].plot(t[:200], sig[:200], color="k", lw=1); axes[0].set_title("Сигнал во времени: 50 Гц + 120 Гц + шум")
    axes[0].set_xlabel("время, с")
    axes[1].plot(freqs, 2 * np.abs(F) / t.size, color=C[0], lw=1.5)
    axes[1].set_xlim(0, 250); axes[1].set_xlabel("частота, Гц"); axes[1].set_title("Спектр |FFT|: два чётких пика")
    Ff = F.copy(); Ff[np.abs(freqs - 50) > 3] = 0
    axes[2].plot(t[:200], np.fft.irfft(Ff)[:200], color=C[2], lw=2, label="только 50 Гц (фильтр в спектре)")
    axes[2].plot(t[:200], np.sin(2 * np.pi * 50 * t[:200]), "k--", lw=1, label="истинная компонента")
    axes[2].legend(fontsize=8, loc="lower right"); axes[2].set_title("Обратное FFT: шум и 120 Гц убраны")
    save(fig, "fourier")

fourier()